# 04 — Spatial Probability Map

Runs sliding-window inference over a full reef scene to produce a
spatial bleaching probability map.

A reference tile from the same site and date is paired with each
query window. The output is a heatmap overlaid on the RGB composite
showing where the model predicts bleaching.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CKPT_PATH    = "/path/to/model_states/best.pt"
PROJECT_DIR  = "/path/to/ct_classifier"
FIGURES_DIR  = "figures"

# Full-scene GeoTIFF to run inference over (query image)
QUERY_TIF    = "/path/to/planet_superdove_landmasked/cheeca_flkeys/bleached/tiled_360m/20230806/loc001.tif"
# Healthy reference tile from the same site (paired with every query window)
REF_TIF      = "/path/to/planet_superdove_landmasked/cheeca_flkeys/healthy/tiled_360m/20210801/loc001.tif"

TILE_SIZE    = 120    # must match model training size
STRIDE       = 60     # overlap between windows (pixels)
BATCH_SIZE   = 16

In [ ]:
import os, sys
import yaml
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import torch
import torch.nn.functional as F

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

from ct_classifier.model import CustomResNet
from utils.viz import to_display_rgb, percentile_stretch

os.makedirs(FIGURES_DIR, exist_ok=True)

cfg_path = os.path.join(os.path.dirname(CKPT_PATH), "config.yaml")
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
norm_factor = float(cfg["normalization_factor"])

model = CustomResNet(cfg["num_classes"], cfg["layers"])
state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state["model"])
model.to(device).eval()
print("Model loaded. Device:", device)

In [ ]:
# ── Read and normalize both scenes ────────────────────────────────────────────
def read_tif(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)   # [8, H, W]
    arr = np.nan_to_num(arr, nan=0.0)
    valid = np.isfinite(arr).all(axis=0).astype(np.float32)  # [H, W]
    return arr, valid

ref_arr,   ref_mask   = read_tif(REF_TIF)
query_arr, query_mask = read_tif(QUERY_TIF)

H, W = query_arr.shape[1], query_arr.shape[2]
print(f"Scene size: {H} x {W}")

In [ ]:
# ── Sliding-window inference ──────────────────────────────────────────────────
prob_map   = np.zeros((H, W), dtype=np.float32)
count_map  = np.zeros((H, W), dtype=np.float32)

windows = [
    (top, left)
    for top  in range(0, H - TILE_SIZE + 1, STRIDE)
    for left in range(0, W - TILE_SIZE + 1, STRIDE)
]

print(f"Running inference on {len(windows)} windows...")

batch_tiles, batch_coords = [], []

def run_batch(tiles, coords):
    x = torch.stack(tiles).to(device)   # [B, 18, T, T]
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[:, 1].cpu().numpy()
    for prob, (t, l) in zip(probs, coords):
        prob_map[t:t+TILE_SIZE, l:l+TILE_SIZE]  += prob
        count_map[t:t+TILE_SIZE, l:l+TILE_SIZE] += 1

for top, left in windows:
    ref_crop   = ref_arr[:,   top:top+TILE_SIZE, left:left+TILE_SIZE]
    qry_crop   = query_arr[:, top:top+TILE_SIZE, left:left+TILE_SIZE]
    ref_m_crop = ref_mask[    top:top+TILE_SIZE, left:left+TILE_SIZE]
    qry_m_crop = query_mask[  top:top+TILE_SIZE, left:left+TILE_SIZE]

    x = np.concatenate([ref_crop, qry_crop], axis=0) / norm_factor
    m = np.stack([ref_m_crop, qry_m_crop], axis=0)
    tile = torch.from_numpy(np.concatenate([x, m], axis=0))  # [18, T, T]

    batch_tiles.append(tile)
    batch_coords.append((top, left))

    if len(batch_tiles) == BATCH_SIZE:
        run_batch(batch_tiles, batch_coords)
        batch_tiles, batch_coords = [], []

if batch_tiles:
    run_batch(batch_tiles, batch_coords)

count_map = np.where(count_map == 0, 1, count_map)
prob_map /= count_map
print("Done. Prob range:", prob_map.min().round(3), "–", prob_map.max().round(3))

In [ ]:
# ── Plot ──────────────────────────────────────────────────────────────────────
rgb = percentile_stretch(to_display_rgb(torch.from_numpy(query_arr)))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(rgb)
axes[0].set_title("Query scene (RGB)")
axes[0].axis("off")

im = axes[1].imshow(prob_map, cmap="RdYlGn_r", vmin=0, vmax=1)
axes[1].set_title("Bleaching probability map")
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="P(bleached)")

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "spatial_prob_map.png"), dpi=150, bbox_inches="tight")
plt.show()